In [60]:
import pandas as pd
import gc

In [61]:
df = pd.read_csv('../data/fines.csv')

## Итерации - сравнение производительности

In [62]:
def calculate_value(row):
    try:
        if row['Refund'] == 0:
            return 0
        return row['Fines'] / row['Refund'] * row['Year']
    except:
        return 0

## 1. Цикл с iloc

In [63]:
def with_iloc_loop(df_input):
    results = []
    for i in range(len(df_input)):
        refund = df_input.iloc[i]['Refund']
        if refund == 0:
            results.append(0)
        else:
            results.append(df_input.iloc[i]['Fines'] / refund * df_input.iloc[i]['Year'])
    return results

df_test = df.copy()
%timeit -n 3 -r 2
df_test['calculated'] = with_iloc_loop(df_test)
print("Результат с iloc:")
print(df_test[['Fines', 'Refund', 'Year', 'calculated']].head())

Результат с iloc:
    Fines  Refund  Year  calculated
0  3200.0     2.0  1989   3182400.0
1  6500.0     1.0  1995  12967500.0
2  2100.0     1.0  1984   4166400.0
3  2000.0     2.0  2015   2015000.0
4  5700.0     1.0  2014  11479800.0


## 2. Использование iterrows()

In [64]:
def with_iterrows(df_input):
    results = []
    for idx, row in df_input.iterrows():
        if row['Refund'] == 0:
            results.append(0)
        else:
            results.append(row['Fines'] / row['Refund'] * row['Year'])
    return results

df_test = df.copy()
%timeit -n 3 -r 2
df_test['calculated'] = with_iterrows(df_test)
print("Результат с iterrows:")
print(df_test[['Fines', 'Refund', 'Year', 'calculated']].head())

Результат с iterrows:
    Fines  Refund  Year  calculated
0  3200.0     2.0  1989   3182400.0
1  6500.0     1.0  1995  12967500.0
2  2100.0     1.0  1984   4166400.0
3  2000.0     2.0  2015   2015000.0
4  5700.0     1.0  2014  11479800.0


## 3. Использование apply() с лямбда-функцией

In [65]:
df_test = df.copy()
%timeit -n 3 -r 2
df_test['calculated'] = df_test.apply(lambda row: row['Fines'] / row['Refund'] * row['Year'] if row['Refund'] != 0 else 0, axis=1)
print("Результат с apply:")
print(df_test[['Fines', 'Refund', 'Year', 'calculated']].head())

Результат с apply:
    Fines  Refund  Year  calculated
0  3200.0     2.0  1989   3182400.0
1  6500.0     1.0  1995  12967500.0
2  2100.0     1.0  1984   4166400.0
3  2000.0     2.0  2015   2015000.0
4  5700.0     1.0  2014  11479800.0


In [66]:
def with_series(df_input):
    return df_input['Fines'] / df_input['Refund'] * df_input['Year']

df_test = df.copy()
%timeit -n 3 -r 2
df_test['calculated'] = with_series(df_test)
df_test.loc[df_test['Refund'] == 0, 'calculated'] = 0
print("Результат с Series:")
print(df_test[['Fines', 'Refund', 'Year', 'calculated']].head())

Результат с Series:
    Fines  Refund  Year  calculated
0  3200.0     2.0  1989   3182400.0
1  6500.0     1.0  1995  12967500.0
2  2100.0     1.0  1984   4166400.0
3  2000.0     2.0  2015   2015000.0
4  5700.0     1.0  2014  11479800.0


## 5. Использование .values

In [67]:
def with_values(df_input):
    fines = df_input['Fines'].values
    refunds = df_input['Refund'].values
    years = df_input['Year'].values
    
    # Создаем массив результатов
    result = []
    for i in range(len(fines)):
        if refunds[i] != 0 and not pd.isna(refunds[i]):
            result.append(fines[i] / refunds[i] * years[i])
        else:
            result.append(0)
    
    return result

df_test = df.copy()
%timeit -n 3 -r 2
df_test['calculated'] = with_values(df_test)
print("Результат с .values:")
print(df_test[['Fines', 'Refund', 'Year', 'calculated']].head())

Результат с .values:
    Fines  Refund  Year  calculated
0  3200.0     2.0  1989   3182400.0
1  6500.0     1.0  1995  12967500.0
2  2100.0     1.0  1984   4166400.0
3  2000.0     2.0  2015   2015000.0
4  5700.0     1.0  2014  11479800.0


## Индексация

In [68]:
print("Поиск без индекса:")
car_number = "O136HO197RUS"

%timeit -n 100 -r 5
df[df['CarNumber'] == car_number]

Поиск без индекса:


,CarNumber,Refund,Fines,Make,Model,Year
715,O136HO197RUS,2.0,7800.0,Toyota,Corolla,1999
896,O136HO197RUS,2.0,7800.0,Toyota,Corolla,1996


In [69]:
df_indexed = df.copy()
df_indexed.set_index('CarNumber', inplace=True)

print("\nПоиск с индексом:")
%timeit -n 100 -r 5
df_indexed.loc[car_number]


Поиск с индексом:


,Refund,Fines,Make,Model,Year
CarNumber,,,,,
O136HO197RUS,2.0,7800.0,Toyota,Corolla,1999
O136HO197RUS,2.0,7800.0,Toyota,Corolla,1996


## Понижение разрядности типов (Downcasting)

In [70]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 927 entries, 0 to 926
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  927 non-null    str    
 1   Refund     927 non-null    float64
 2   Fines      927 non-null    float64
 3   Make       927 non-null    str    
 4   Model      918 non-null    str    
 5   Year       927 non-null    int64  
dtypes: float64(2), int64(1), str(3)
memory usage: 174.3 KB


In [71]:
optimized_df = df.copy()

In [72]:
float_columns = optimized_df.select_dtypes(include=['float64']).columns
for col in float_columns:
    optimized_df[col] = optimized_df[col].astype('float32')
    print(f"  {col}: float64 -> float32")

  Refund: float64 -> float32
  Fines: float64 -> float32


In [73]:
int_columns = optimized_df.select_dtypes(include=['int64']).columns
for col in int_columns:
    col_min = optimized_df[col].min()
    col_max = optimized_df[col].max()
    
    if col_min >= 0:
        if col_max <= 255:
            optimized_df[col] = optimized_df[col].astype('uint8')
            print(f"  {col}: int64 -> uint8")
        elif col_max <= 65535:
            optimized_df[col] = optimized_df[col].astype('uint16')
            print(f"  {col}: int64 -> uint16")
        elif col_max <= 4294967295:
            optimized_df[col] = optimized_df[col].astype('uint32')
            print(f"  {col}: int64 -> uint32")
        else:
            optimized_df[col] = optimized_df[col].astype('uint64')
            print(f"  {col}: int64 -> uint64")
    else:
        if col_min >= -128 and col_max <= 127:
            optimized_df[col] = optimized_df[col].astype('int8')
            print(f"  {col}: int64 -> int8")
        elif col_min >= -32768 and col_max <= 32767:
            optimized_df[col] = optimized_df[col].astype('int16')
            print(f"  {col}: int64 -> int16")
        elif col_min >= -2147483648 and col_max <= 2147483647:
            optimized_df[col] = optimized_df[col].astype('int32')
            print(f"  {col}: int64 -> int32")
        else:
            print(f"  {col}: остается int64")

  Year: int64 -> uint16


In [74]:
optimized_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 927 entries, 0 to 926
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  927 non-null    str    
 1   Refund     927 non-null    float32
 2   Fines      927 non-null    float32
 3   Make       927 non-null    str    
 4   Model      918 non-null    str    
 5   Year       927 non-null    uint16 
dtypes: float32(2), str(3), uint16(1)
memory usage: 161.6 KB


## Категории

In [75]:
categorical_df = optimized_df.copy()

object_columns = categorical_df.select_dtypes(include=['object']).columns
print("Преобразование object -> category:")
for col in object_columns:
    categorical_df[col] = categorical_df[col].astype('category')
    print(f"  {col}: object -> category")

Преобразование object -> category:
  CarNumber: object -> category
  Make: object -> category
  Model: object -> category


/var/folders/q7/70k5z3797l7953_1tz5w6xcr0000gn/T/ipykernel_5537/3689972392.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = categorical_df.select_dtypes(include=['object']).columns


In [76]:
categorical_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 927 entries, 0 to 926
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   CarNumber  927 non-null    category
 1   Refund     927 non-null    float32 
 2   Fines      927 non-null    float32 
 3   Make       927 non-null    category
 4   Model      918 non-null    category
 5   Year       927 non-null    uint16  
dtypes: category(3), float32(2), uint16(1)
memory usage: 45.9 KB


## Очистка памяти

In [77]:
del df
gc.collect()
%reset_selective -f df

In [78]:
df

NameError: name 'df' is not defined